In [ ]:
import sys, os, subprocess
subprocess.run([sys.executable,'-m','pip','install','-q','onnxruntime'],check=False)
print('ort ready')


# BirdCLEF 2026 Inference v40 — Perch + BirdNET Dual-Stream

Runs both Perch ONNX and BirdNET ONNX per 5-s window, concatenates embeddings,
feeds through v40 dual-stream GRU.

| Version | LB |
|---------|----|  
| v30 GRU | **0.875** |
| v38 temp scaling | 0.875 |
| v39 focal | 0.863 |
| **v40 dual-stream** | ? |

**Kaggle inputs required:**
1. `birdclef-2026`
2. `chiragggg/birdclef-2026-perch-onnx`
3. BirdNET ONNX dataset (same as used in embed notebook)
4. `chiragggg/birdclef-2026-perch-weights-v40`


In [ ]:
import os, warnings, gc
from pathlib import Path
import numpy as np, pandas as pd, soundfile as sf, librosa, onnxruntime as ort
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn
from torch.cuda.amp import autocast
from tqdm import tqdm
warnings.filterwarnings('ignore')

CFG = dict(
    folds=5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    perch_sr=32000, perch_seconds=5, perch_emb_dim=1536, perch_batch=16,
    birdnet_sr=48000, birdnet_seconds=3, birdnet_emb_dim=1024, birdnet_batch=16,
    gru_hidden=512, gru_layers=2, gru_dropout=0.3,
    gauss_sigma=1.0,
)
CFG['perch_target']   = CFG['perch_sr']   * CFG['perch_seconds']
CFG['birdnet_target'] = CFG['birdnet_sr'] * CFG['birdnet_seconds']
CFG['concat_dim']     = CFG['perch_emb_dim'] + CFG['birdnet_emb_dim']
device = torch.device(CFG['device'])
torch.set_num_threads(os.cpu_count() or 4)
print(f'Device: {device}  concat_dim={CFG["concat_dim"]}')


In [ ]:
def _fe(*c): return next((p for p in c if os.path.exists(p)), c[0])

TAXONOMY_CSV = _fe('/kaggle/input/birdclef-2026/taxonomy.csv',
                   '/kaggle/input/competitions/birdclef-2026/taxonomy.csv')
TEST_AUDIO   = _fe('/kaggle/input/birdclef-2026/test_soundscapes',
                   '/kaggle/input/competitions/birdclef-2026/test_soundscapes')
SAMPLE_SUB   = _fe('/kaggle/input/birdclef-2026/sample_submission.csv',
                   '/kaggle/input/competitions/birdclef-2026/sample_submission.csv')

GRU_CKPT_DIR = _fe('/kaggle/input/birdclef-2026-perch-weights-v40',
                   '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-weights-v40',
                   '/kaggle/working')

PERCH_ONNX_PATH = None
for _c in [
    '/kaggle/input/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/datasets/chiragggg/birdclef-2026-perch-onnx/perch_v2_cpu.onnx',
    '/kaggle/input/perch-onnx-for-birdclef2026/perch_v2_cpu.onnx',
]:
    if os.path.exists(_c): PERCH_ONNX_PATH = _c; break

BIRDNET_ONNX_PATH = None
for _c in [
    '/kaggle/input/birdnet-analyzer-onnx/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-analyzer/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-global-6k-v24/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet-onnx/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
    '/kaggle/input/birdnet/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx',
]:
    if os.path.exists(_c): BIRDNET_ONNX_PATH = _c; break

# *** Set manually if auto-detect fails: ***
# BIRDNET_ONNX_PATH = '/kaggle/input/YOUR-DATASET/BirdNET_GLOBAL_6K_V2.4_Model_FP32.onnx'

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
species     = taxonomy_df['primary_label'].astype(str).tolist()
n_classes   = len(species)
sp_idx      = {l: i for i, l in enumerate(species)}
print(f'Species          : {n_classes}')
print(f'GRU_CKPT_DIR     : {GRU_CKPT_DIR}')
print(f'PERCH_ONNX_PATH  : {PERCH_ONNX_PATH}')
print(f'BIRDNET_ONNX_PATH: {BIRDNET_ONNX_PATH}')


In [ ]:
# PerchGRUDual — must match training architecture exactly
class PerchGRUDual(nn.Module):
    def __init__(self, n_classes, concat_dim, hidden=512, n_layers=2, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LayerNorm(concat_dim),
            nn.Linear(concat_dim, 512),
            nn.GELU(),
        )
        self.gru = nn.GRU(512, hidden, n_layers, batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(0.2),
            nn.Linear(hidden * 2, n_classes),
        )
    def forward(self, x):
        single = (x.dim() == 2)
        if single: x = x.unsqueeze(1)
        h, _ = self.gru(self.proj(x))
        out   = self.head(h)
        return out.squeeze(1) if single else out

print('PerchGRUDual defined')


In [ ]:
def _load_gru(names, ckpt_dir):
    ms = []
    for n in names:
        p = Path(ckpt_dir) / n
        if not p.exists(): print(f'  MISSING: {p}'); continue
        m = PerchGRUDual(n_classes, CFG['concat_dim'],
                         CFG['gru_hidden'], CFG['gru_layers'], CFG['gru_dropout']).to(device)
        m.load_state_dict(torch.load(p, map_location=device, weights_only=True), strict=True)
        m.eval()
        ms.append(m)
        print(f'  OK {n}')
    return ms

print('Loading v40 checkpoints...')
gru_models = _load_gru(
    [f'perch_gru_v40_fold{i}.pt' for i in range(CFG['folds'])],
    GRU_CKPT_DIR,
)
print(f'Loaded: {len(gru_models)}/5')


In [ ]:
# === PERCH ONNX SESSION ===
_psess = None; _pinp = None; _peidx = 0; _perch_ok = False
if PERCH_ONNX_PATH:
    try:
        opts = ort.SessionOptions()
        opts.intra_op_num_threads = os.cpu_count() or 4
        _psess = ort.InferenceSession(PERCH_ONNX_PATH, sess_options=opts,
                                       providers=['CPUExecutionProvider'])
        _pinp  = _psess.get_inputs()[0].name
        _pout  = [o.name for o in _psess.get_outputs()]
        _peidx = next((i for i,o in enumerate(_psess.get_outputs()) if o.shape and o.shape[-1]==1536), 0)
        _t = _psess.run(None, {_pinp: np.zeros((1,CFG['perch_target']),np.float32)})
        _e = _t[_peidx]; _e = _e.mean(1) if _e.ndim==3 else _e
        assert _e.shape[-1] == 1536
        _perch_ok = True
        print(f'Perch ONNX OK: emb={_e.shape}')
    except Exception as ex:
        print(f'Perch ONNX ERROR: {ex}')
else:
    print('Perch ONNX not found')

# === BIRDNET ONNX SESSION ===
_bsess = None; _binp = None; _beidx = 0; _bnet_ok = False; _bnet_dim = CFG['birdnet_emb_dim']
if BIRDNET_ONNX_PATH:
    try:
        opts2 = ort.SessionOptions()
        opts2.intra_op_num_threads = os.cpu_count() or 4
        _bsess = ort.InferenceSession(BIRDNET_ONNX_PATH, sess_options=opts2,
                                       providers=['CPUExecutionProvider'])
        _binp  = _bsess.get_inputs()[0].name
        _beidx = next((i for i,o in enumerate(_bsess.get_outputs()) if o.shape and o.shape[-1]==1024), 0)
        _t2 = _bsess.run(None, {_binp: np.zeros((1,CFG['birdnet_target']),np.float32)})
        _e2 = _t2[_beidx]; _e2 = _e2.mean(1) if _e2.ndim==3 else _e2
        _bnet_dim = int(_e2.shape[-1])
        CFG['birdnet_emb_dim'] = _bnet_dim
        CFG['concat_dim']      = CFG['perch_emb_dim'] + _bnet_dim
        _bnet_ok = True
        print(f'BirdNET ONNX OK: emb={_e2.shape}  concat_dim={CFG["concat_dim"]}')
    except Exception as ex:
        print(f'BirdNET ONNX ERROR: {ex}')
else:
    print('BirdNET ONNX not found — BirdNET stream will be zeros')

print(f'perch_ok={_perch_ok}  bnet_ok={_bnet_ok}')


In [ ]:
_amp = (device.type == 'cuda')


def _perch_embs(y_32k, ends):
    if not _perch_ok: return {}
    clips = []
    for es in ends:
        e0 = int(es * CFG['perch_sr']); s0 = max(0, e0 - CFG['perch_target'])
        c  = y_32k[s0:e0]
        if len(c) < CFG['perch_target']: c = np.pad(c,(0,CFG['perch_target']-len(c)))
        clips.append(c)
    all_embs = []
    for bi in range(0,len(clips),CFG['perch_batch']):
        B = np.stack(clips[bi:bi+CFG['perch_batch']])
        o = _psess.run(None,{_pinp:B})[_peidx]
        if o.ndim==3: o=o.mean(1)
        all_embs.append(o.astype(np.float32))
    return dict(zip(ends, np.vstack(all_embs)))


def _birdnet_embs(y_48k, ends):
    if not _bnet_ok: return {}
    clips = []
    for es in ends:
        e0 = int(es * CFG['birdnet_sr']); s0 = max(0, e0 - CFG['birdnet_target'])
        c  = y_48k[s0:e0]
        if len(c) < CFG['birdnet_target']: c = np.pad(c,(0,CFG['birdnet_target']-len(c)))
        clips.append(c)
    all_embs = []
    for bi in range(0,len(clips),CFG['birdnet_batch']):
        B = np.stack(clips[bi:bi+CFG['birdnet_batch']])
        o = _bsess.run(None,{_binp:B})[_beidx]
        if o.ndim==3: o=o.mean(1)
        all_embs.append(o.astype(np.float32))
    return dict(zip(ends, np.vstack(all_embs)))


def predict_v40(path, ends):
    T_win = len(ends)
    if not gru_models or not _perch_ok:
        return np.full((T_win, n_classes), 0.5, np.float32)

    try:
        y_raw, sr = sf.read(path, always_2d=False)
        if y_raw.ndim == 2: y_raw = y_raw.mean(1)
        y_raw = y_raw.astype(np.float32)
    except Exception as e:
        print(f'[W] read {path}: {e}')
        return np.full((T_win, n_classes), 0.5, np.float32)

    # Resample once to each SR
    y_32k = librosa.resample(y_raw, orig_sr=sr, target_sr=CFG['perch_sr'])   if sr!=CFG['perch_sr']   else y_raw
    y_48k = librosa.resample(y_raw, orig_sr=sr, target_sr=CFG['birdnet_sr']) if sr!=CFG['birdnet_sr'] else y_raw

    pe = _perch_embs(y_32k, ends)    # {end_secs: (1536,)}
    be = _birdnet_embs(y_48k, ends)  # {end_secs: (1024,)}

    # Build (1, T, concat_dim) input tensor
    rows = []
    for e in ends:
        pv = pe.get(e, np.zeros(CFG['perch_emb_dim'],   np.float32))
        bv = be.get(e, np.zeros(CFG['birdnet_emb_dim'], np.float32))
        rows.append(np.concatenate([pv, bv]))
    seq = torch.from_numpy(np.stack(rows)).float().unsqueeze(0).to(device)

    preds = []
    for m in gru_models:
        with torch.inference_mode(), autocast(enabled=_amp):
            logits = m(seq).float()[0].cpu().numpy()
        preds.append(1.0 / (1.0 + np.exp(-logits)))

    p = np.mean(preds, axis=0)
    if T_win > 1 and CFG['gauss_sigma'] > 0:
        p = gaussian_filter1d(p.astype(np.float64), sigma=CFG['gauss_sigma'], axis=0).astype(np.float32)
    return p


print(f'predict_v40 defined  folds={len(gru_models)}  perch_ok={_perch_ok}  bnet_ok={_bnet_ok}')


In [ ]:
sub = pd.read_csv(SAMPLE_SUB).copy()
sub['_sc'] = sub['row_id'].str.rsplit('_', n=1).str[0]
print(f'Rows: {len(sub)}')

row_ids=[]; probs_list=[]; n_miss=0; n_err=0

for sc, grp in tqdm(sub.groupby('_sc'), desc='soundscapes', unit='f'):
    rids = [str(r) for r in grp['row_id']]
    ap = None
    for ext in ['.ogg','.wav','.flac']:
        c = Path(TEST_AUDIO)/f'{sc}{ext}'
        if c.exists(): ap=str(c); break
    if ap is None:
        n_miss+=1; row_ids.extend(rids)
        probs_list.append(np.full((len(rids),n_classes),0.5,np.float32)); continue
    try:
        ends=[int(r.rsplit('_',1)[-1]) for r in rids]
    except Exception:
        n_err+=1; row_ids.extend(rids)
        probs_list.append(np.full((len(rids),n_classes),0.5,np.float32)); continue
    try:
        p=predict_v40(ap,ends); row_ids.extend(rids); probs_list.append(p)
    except Exception as e:
        n_err+=1; print(f'ERR {sc}: {e}'); row_ids.extend(rids)
        probs_list.append(np.full((len(rids),n_classes),0.5,np.float32))

print(f'Done  missing={n_miss}  errors={n_err}')


In [ ]:
mat = np.concatenate(probs_list, axis=0)
print(f'mean={mat.mean():.4f}  std={mat.std():.4f}')

sub_df = pd.DataFrame(mat, columns=species)
sub_df.insert(0,'row_id',row_ids)
cols = pd.read_csv(SAMPLE_SUB, nrows=0).columns.tolist()
sub_df = sub_df[cols]
sub_df.to_csv('/kaggle/working/submission.csv', index=False)
print(f'Saved  shape={sub_df.shape}')
print(f'v40  folds={len(gru_models)}  concat_dim={CFG["concat_dim"]}')
sub_df.head(3)
